### Работа с гиперпараметрами моделей

В машинном обучении успешное построение модели зависит не только от выбора алгоритма, но и от правильной настройки его параметров. Чтобы модель показывала высокую точность и хорошо обобщала закономерности в данных, необходимо правильно настроить гиперпараметры.

Подбор гиперпараметров — это процесс поиска таких значений гиперпараметров, при которых модель достигает наилучших результатов на валидационных данных. Для начала давайте разберемся в определениях.

Параметры (parameters) — это внутренние переменные, которые модель оценивает в процессе обучения на основе данных.

Они подбираются автоматически методом оптимизации (например, градиентным спуском). Параметры определяют конкретную структуру модели после обучения.

Примеры параметров:

В линейной регрессии — коэффициенты (w) и смещение (b) в уравнении регрессии.
В логистической регрессии — коэффициенты (w) и смещение (b) в уравнении.
В SVM — вектора опорных точек.
В деревьях решений и случайных лесах — структура дерева (какие признаки используются для разбиения).
Гиперпараметры (hyperparameters) — это параметры, которые не обучаются автоматически, а их нужно задавать вручную до начала обучения модели.

Они управляют процессом обучения и сложностью модели. Подбор гиперпараметров осуществляется отдельно от обучения модели.

Алгоритм ML	Примеры гиперпараметров
Метод ближайших соседей (KNN)	Количество соседей (n_neighbors), метод взвешивания (weights), метрика расстояния (metrics) и пр.
Дерево решений (Decision Tree)	Максимальная глубина дерева (max_depth), минимальное количество выборок в листе (min_samples_leaf) и пр.
Случайный лес (Random Forest)	Количество деревьев в ансамбле (n_estimators), максимальная глубина дерева (max_depth) и пр.
Метод опорных векторов (SVM)	Параметр регуляризации (C), параметр ядра (kernel), коэффициент для RBF-ядра (gamma) и пр.
Они используются для управления процессом обучения, поэтому подбор оптимальных гиперпараметров — очень важный этап в построении ML-моделей, позволяющий повысить точность, а также бороться с переобучением.

Методы подбора гиперпараметров
Ручной подбор (Manual Search) — подбор гиперпараметров на основе интуиции и экспериментов.
Подходит для небольших моделей и когда есть понимание влияния гиперпараметров.
Минусы: трудоемкость, сложность для сложных моделей с большим количеством гиперпараметров.
Grid Search (перебор по сетке) — это перебор всех возможных комбинаций гиперпараметров из заданной сетки.
Подходит для небольшого числа гиперпараметров из-за экспоненциального роста числа комбинаций.
Минусы: медленно при большом количестве параметров.
Random Search (случайный поиск) — это случайный перебор комбинаций гиперпараметров в пределах заданных диапазонов.
Позволяет охватить больше гиперпараметров за меньшее время.
Работает быстрее, чем Grid Search при большом количестве гиперпараметров.
Байесовская оптимизация (Bayesian Optimization) — это метод, который строит вероятностную модель зависимости гиперпараметров от качества модели. На каждом шаге он обновляет модель и делает выбор параметров на основе предсказаний.
Эффективен при большом количестве гиперпараметров.
Есть и другие способы, но остановимся и рассмотрим три самых популярных подхода.

Для примера возьмем все тот же датасет с Kaggle, который содержит информацию о потреблении электричества на одной из подстанций Москвы до 2008 года включительно. А в рамках соревнования требуется построить предсказание потребления электричества на первую половину 2009 года.

In [1]:
import pandas as pd

df = pd.read_csv(r"C:\IDE\2 семестр\Классическое МО\data\electricity-consumption\train.csv")

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from TimeFeaturesExtractor import TimeFeaturesExtractor
import numpy as np
from TrigonometricFeaturesExtractor import TrigonometricFeaturesExtractor
import math

In [3]:
pipeline_preprocessing = Pipeline([
    ('time_features', TimeFeaturesExtractor()), # призанки даты и времени
    ('trigonometric_features', TrigonometricFeaturesExtractor()), # Тригонометрические признаки
])

# просто удаляем пропуски
df = df[~df['total'].isna()]
df['total'] = df['total'].astype('int64')

# Преобразуем строковый столбец 'datetime' в формат datetime
df['datetime'] = pd.to_datetime(df['datetime'], format="%d.%m.%Y %H:%M:%S")

# Разделяем данные на train и test, извлекая год прямо из 'datetime'
train = df[df['datetime'].dt.year != 2008]
test = df[df['datetime'].dt.year == 2008]

# Удалим ненужные колонки (например, целевую переменную) из обучающих данных
X_train = train.drop(columns=['total'])
y_train = train['total']

'''# Применим пайплайн и обучим модель
pipeline_preprocessing.fit(X_train, y_train)'''

# Удалим ненужные колонки из тестовых данных
X_test = test.drop(columns=['total'])
y_test = test['total']

# Применим пайплайн к тестовым данным (переход от fit к transform)
X_train_transformed = pipeline_preprocessing.fit_transform(X_train)
X_test_transformed = pipeline_preprocessing.transform(X_test)

In [4]:
'''Теперь строим итоговый Pipeline с учетом обучения модели:'''

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# Полный пайплайн с моделью
full_pipeline = Pipeline([
    ('preprocessing', pipeline_preprocessing),  # Готовый пайплайн для обработки данных
    ('scaler', StandardScaler()),               # Масштабирование данных
    ('model', RandomForestRegressor())          # Заглушка для модели
])

### Grid Search (перебор по сетке)
GridSearchCV — это метод из библиотеки scikit-learn для исчерпывающего перебора всех возможных комбинаций гиперпараметров, заданных в сетке (grid). Он автоматически проводит кросс-валидацию и выбирает лучшую комбинацию гиперпараметров на основе заданной метрики.
Как работает GridSearchCV:

Задаем сетку гиперпараметров — указываем, какие значения для гиперпараметров нужно протестировать.
Кросс-валидация — данные разбиваются на несколько блоков (fold'ов), и модель обучается на всех блоках, кроме одного, который используется для проверки. Этот процесс повторяется для всех комбинаций гиперпараметров.
Оценка качества — после каждой итерации фиксируется значение метрики на валидационном блоке.
Выбор лучшей модели — по итогам поиска выбирается комбинация гиперпараметров с лучшим значением метрики.
Когда задаете сетку гиперпараметров, учитывайте следующее:

Гиперпараметры должны соответствовать модели, которую вы используете. Например, n_estimators используется в случайном лесе, но не в SVM.
Чем больше вариантов в сетке, тем больше комбинаций и тем выше вычислительная сложность.

In [5]:
# Сетка гиперпараметров для Random Forest
param_grid = {
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [3, 5, 7],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

Здесь в сетке возможных параметров, вы самостоятельно указываете значения для перебора, но важно понимать, что общее количество комбинаций гиперпараметров напрямую зависит от числа значений, которые вы перечислили для каждого параметра.

Общее количество комбинаций можно рассчитать по формуле:

К
о
л
в
о
к
о
м
б
и
н
а
ц
и
й

где  — количество вариантов значений для каждого гиперпараметра.

Для нашей сетки получится 3 * 3 * 2 * 2 = 36 комбинаций гиперпараметров для случайного леса. Если используется кросс-валидация cv=5, то модель обучится 36 × 5 = 180 раз.

Создаем объект GridSearchCV:

estimator — модель, которую мы хотим настроить или Pipeline, который содержит модель.
param_grid — сетка гиперпараметров.
cv — количество фолдов для кросс-валидации.
scoring — метрика для оценки качества модели.
n_jobs — количество потоков для параллельных вычислений (-1 — использовать все).

In [6]:
# Подбор гиперпараметров через GridSearchCV
search = GridSearchCV(full_pipeline,
                      param_grid,
                      cv=5,
                      scoring='neg_mean_squared_error')
# Обучаем алгоритм подбора гиперпараметров
search.fit(X_train, y_train)

# Лучшие параметры
print(f"Best params: {search.best_params_}")
print(f"Best score: {search.best_score_}")

Best params: {'model__max_depth': 7, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Best score: -208517601.26076663


In [ ]:
'''Так как комбинаций много, при текущей настройке алгоритм будет обучаться около 15 минут.
Получаем результаты.

Лучшая комбинация гиперпараметров:'''
search.best_params_
'''Best params: {'model__max_depth': 7, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Лучшая оценка на кросс-валидации:'''
search.best_score_
'''Best score: -208818434.0967043'''
'''Лучшая модель с настроенными параметрами:'''
best_model = search.best_estimator_
'''Оценка на тестовой выборке:'''

y_pred = best_model.predict(X_test)
print("Test MSE:", mean_squared_error(y_test, y_pred))

Random Search (случайный поиск)
RandomizedSearchCV — это метод из библиотеки scikit-learn для случайного поиска наилучшей комбинации гиперпараметров. В отличие от GridSearchCV, где происходит полный перебор всех возможных комбинаций, RandomizedSearchCV выбирает случайное подмножество из сетки гиперпараметров и тестирует только его.
Как работает RandomizedSearchCV:

Задаем сетку гиперпараметров — так же, как в GridSearchCV, задаем диапазоны значений, которые хотим протестировать.
Случайный выбор гиперпараметров — случайным образом выбираются комбинации из заданной сетки.
Кросс-валидация — для каждой выбранной случайной комбинации гиперпараметров выполняется кросс-валидация.
Оценка и выбор лучшей комбинации — после всех прогонов выбирается лучшая комбинация по заданной метрике.
Метод	Как ищет
гиперпараметры	Количество
комбинаций	Подходит для
GridSearchCV	Полный перебор всех комбинаций	Очень большое (растет экспоненциально)	Небольшие сетки
RandomizedSearchCV	Случайный перебор фиксированного количества комбинаций	Управляемое, можно задать вручную	Большие сетки, сложные модели
Создаем объект RandomizedSearchCV:

n_iter — количество случайных комбинаций, которые мы хотим протестировать.
cv — количество фолдов для кросс-валидации.
scoring — метрика для оценки качества модели..
n_jobs — количество потоков для параллельных вычислений.

In [ ]:
# Сетка гиперпараметров для Random Forest
param_dist = {
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [3, 5, 7],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Определяем модель
model = RandomForestClassifier()

# Создаем объект RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_dist,  # Диапазон параметров
    n_iter=20,                       # 20 случайных комбинаций
    cv=5,                            # 5-кратная кросс-валидация
    scoring='accuracy',              # Метрика - точность
    n_jobs=-1,                       # Параллельное выполнение
    random_state=42                 # Фиксируем сид для воспроизводимости
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
'''Получаем результаты.

Лучшая комбинация гиперпараметров:'''
random_search.best_params_
'''Лучшая оценка на кросс-валидации:'''
random_search.best_score_
'''Лучшая модель с настроенными параметрами:'''
best_model = random_search.best_estimator_
'''А вот оценка на тестовой выборке:'''

y_pred = best_model.predict(X_test)
print("Test MSE:", mean_squared_error(y_test, y_pred))

### Байесовская оптимизация (Bayesian Optimization)
Байесовская оптимизация — это продвинутый метод для поиска наилучшей комбинации гиперпараметров модели на основе построения вероятностной модели функции потерь (ошибки). Она использует информацию о предыдущих запусках для более эффективного поиска новых комбинаций гиперпараметров.
В отличие от GridSearchCV и RandomizedSearchCV, которые «слепо» перебирают параметры, байесовская оптимизация пытается умно предсказывать, какие гиперпараметры стоит протестировать, основываясь на предыдущих результатах.

Суть Байесовской оптимизации состоит в следующем. Создается априорная модель функции потерь на основе начальных данных (случайных запусков). На основе этой модели строится аппроксимация функции потерь.

Далее используется стратегия Exploration vs. Exploitation для поиска следующих параметров:

Exploration — чтобы исследовать новые области.
Exploitation — чтобы углубляться в уже известные хорошие результаты.
После каждой итерации аппроксимация обновляется с учетом новых данных. Процесс повторяется до достижения критерия остановки.

Плюсы байесовской оптимизации:

Учитывает результаты предыдущих запусков, следовательно осуществляет более эффективный поиск.
Позволяет работать с непрерывными значениями гиперпараметров.
Эффективно работает при сложных функциях потерь.
Быстрее находит хорошее решение, чем полный перебор.
Минусы байесовской оптимизации:

Не подходит для очень высокоразмерных пространств параметров.
Требует больше ресурсов для построения аппроксимации.
Не всегда находит глобальный минимум.
Пример с использованием библиотеки scikit-optimize:

In [ ]:
from skopt import BayesSearchCV
from sklearn.ensemble import RandomForestClassifier

# Определяем пространство гиперпараметров
param_space = {
    'n_estimators': (10, 200),                  # Количество деревьев (диапазон значений)
    'max_depth': (1, 20),                       # Глубина дерева (диапазон значений)
    'min_samples_split': (2, 10),               # Минимум элементов для разбиения узла
    'min_samples_leaf': (1, 5)                  # Минимум элементов в листе
}

# Создаем модель
model = RandomForestClassifier()

# Оптимизация с помощью Байесовского поиска
bayes_search = BayesSearchCV(
    estimator=model,
    search_spaces=param_space,     # Пространство гиперпараметров
    n_iter=20,                     # Количество итераций
    cv=5,                          # Количество фолдов для кросс-валидации
    scoring='accuracy',            # Метрика качества
    n_jobs=-1,                     # Параллельное выполнение
    random_state=42
)

In [ ]:
'''Обучаем модель подбора наилучших гиперпараметров:
'''
bayes_search.fit(X_train, y_train)
'''Получаем результаты.

Лучшая комбинация гиперпараметров:'''
print(bayes_search.best_params_)
'''Лучшая оценка на кросс-валидации:'''
print(bayes_search.best_score_)
'''Лучшая модель с оптимальными параметрами:'''
best_model = bayes_search.best_estimator_
'''Оценка на тестовой выборке:'''

y_pred = best_model.predict(X_test)
print("Test MSE:", mean_squared_error(y_test, y_pred))
'''Теперь, когда вы научились подбирать наилучшие гиперпараметры и настраивать Pipeline, можно решать практически любые задачи ML.'''